# PaperGuard T9 — DistilBERT LLM-text classifier (Plan C)

Train a small BERT-family classifier that distinguishes **human-written** from
**ChatGPT-written** answers, on the public [HC3 dataset](https://huggingface.co/datasets/Hello-SimpleAI/HC3).
The exported model becomes PaperGuard's **T9** detector (the 41st detector),
a *learned* complement to the statistical T7 (perplexity) and T8 (DetectGPT)
signals.

> **Iron rule (carried into the detector, not the notebook):** T9 output must
> never use verdict language. Every `Finding` ships >= 3 innocent explanations.
> The classifier reports a *probability*, which T9 maps to NOTE / CONCERN /
> SUSPICIOUS tiers — never to "AI-generated, confirmed".

## How to run (free Colab T4)

1. **Runtime -> Change runtime type -> T4 GPU -> Save.**
2. **Runtime -> Run all.**
3. Training takes ~4–8 h on a T4. It is **resumable**: checkpoints are written
   to your Google Drive every epoch, so if Colab disconnects, just *Run all*
   again — it picks up from the last checkpoint.
4. When it finishes, the final cell prints **accuracy** and **LR+** at the
   SUSPICIOUS operating point (target accuracy >= 0.85), and writes the model to
   your Drive. Follow the *Download & integrate* section at the bottom.

No PaperGuard source is needed inside Colab — this notebook is self-contained.
The integration snippet at the end is dropped into the PaperGuard repo *after*
you download the trained model.


## 1. Setup — install dependencies


In [ ]:
# Cell 1 — install. Pinned-ish to versions known to cooperate on Colab T4.
# If a resolver conflict appears, restart runtime (Runtime -> Restart) and
# Run all again; pip state is the usual culprit, not the code.
!pip install -q "transformers>=4.40,<5" "datasets>=2.19,<3" "accelerate>=0.30" "scikit-learn>=1.3" "evaluate>=0.4"
print("deps installed")


In [ ]:
# Cell 1b — confirm the GPU. If this prints 'CPU only', stop and switch the
# runtime to T4 (Runtime -> Change runtime type). Training on CPU is infeasible.
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda, "| torch:", torch.__version__)
else:
    print("CPU only — switch Runtime to T4 GPU before continuing.")


## 2. Mount Google Drive + configuration

Everything durable (checkpoints, final model) is written under
`MyDrive/paperguard_t9/` so a disconnect never loses progress.


In [ ]:
# Cell 2 — mount Drive and define paths. Re-running is safe (idempotent).
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/paperguard_t9")
CKPT_DIR = DRIVE_ROOT / "checkpoints"   # per-epoch resumable checkpoints
EXPORT_DIR = DRIVE_ROOT / "t9_model"    # final HF save_pretrained() bundle
for d in (DRIVE_ROOT, CKPT_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Reproducibility + hyperparameters (single source of truth) ---
SEED = 42
BASE_MODEL = "distilbert-base-uncased"
MAX_LEN = 512            # HC3 answers are long; 512 captures most signal
NUM_EPOCHS = 3
BATCH_SIZE = 16          # fits T4 16 GB at fp16 + seq-len 512
LR = 2e-5
# Probability of the 'LLM' class above which T9 calls a segment SUSPICIOUS.
# High by design: PaperGuard optimizes specificity (few false alarms) over
# sensitivity. LR+ is reported at this exact threshold.
SUSPICIOUS_THRESHOLD = 0.90

import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Drive root:", DRIVE_ROOT)
print("Config:", dict(base=BASE_MODEL, max_len=MAX_LEN, epochs=NUM_EPOCHS,
                       batch=BATCH_SIZE, lr=LR, suspicious=SUSPICIOUS_THRESHOLD))


## 3. Load and flatten HC3

HC3 stores, per question, a list of `human_answers` and a list of
`chatgpt_answers`. We flatten to one row per answer:

| label | meaning |
|---|---|
| `0` | human-written |
| `1` | ChatGPT-written |

Only the `"all"` config is used (all source domains: reddit_eli5, finance,
medicine, open_qa, wiki_csai).


In [ ]:
# Cell 3 — load HC3 and flatten to (text, label) pairs.
from datasets import load_dataset, Dataset

raw = load_dataset("Hello-SimpleAI/HC3", "all")
# HC3 ships a single 'train' split; we make our own val/test below.
hc3 = raw["train"]
print("HC3 rows (question groups):", len(hc3))

LABEL_HUMAN, LABEL_LLM = 0, 1
texts: list[str] = []
labels: list[int] = []

for row in hc3:
    for ans in (row.get("human_answers") or []):
        a = (ans or "").strip()
        if len(a) >= 40:                 # drop near-empty stubs
            texts.append(a)
            labels.append(LABEL_HUMAN)
    for ans in (row.get("chatgpt_answers") or []):
        a = (ans or "").strip()
        if len(a) >= 40:
            texts.append(a)
            labels.append(LABEL_LLM)

n_human = labels.count(LABEL_HUMAN)
n_llm = labels.count(LABEL_LLM)
print(f"flattened answers: {len(texts):,}  (human={n_human:,}  llm={n_llm:,})")


## 4. Stratified split + tokenize

80 / 10 / 10 train / validation / test, stratified on the label so both classes
appear in every split. The split is seeded, so re-running the notebook (or
resuming) gives the *same* held-out test set — LR+ stays comparable run to run.


In [ ]:
# Cell 4 — split (stratified, seeded) then tokenize.
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

tr_txt, tmp_txt, tr_lab, tmp_lab = train_test_split(
    texts, labels, test_size=0.20, random_state=SEED, stratify=labels
)
val_txt, te_txt, val_lab, te_lab = train_test_split(
    tmp_txt, tmp_lab, test_size=0.50, random_state=SEED, stratify=tmp_lab
)
print(f"train={len(tr_txt):,}  val={len(val_txt):,}  test={len(te_txt):,}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def _tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

def _make(split_txt, split_lab):
    ds = Dataset.from_dict({"text": split_txt, "label": split_lab})
    ds = ds.map(_tok, batched=True, remove_columns=["text"])
    return ds

ds_train = _make(tr_txt, tr_lab)
ds_val = _make(val_txt, val_lab)
ds_test = _make(te_txt, te_lab)
print(ds_train)


## 5. Model + metrics

`DistilBertForSequenceClassification`, 2 labels. The metrics function reports
accuracy / precision / recall / F1 at the default 0.5 boundary, **and** LR+ at
the SUSPICIOUS operating point (`p(LLM) >= SUSPICIOUS_THRESHOLD`), which is the
number PaperGuard actually cares about.


In [ ]:
# Cell 5 — model, label maps, and a metrics fn that also computes LR+.
import numpy as np
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

id2label = {0: "human", 1: "llm"}
label2id = {"human": 0, "llm": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, id2label=id2label, label2id=label2id
)

def _softmax(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def _lr_plus(y_true, p_llm, threshold):
    # LR+ = sensitivity / (1 - specificity) at a probability threshold.
    # Returns float('inf') when there are zero false positives (perfect
    # specificity at this threshold) -- reported verbatim, like T6's LR+.
    y_true = np.asarray(y_true)
    pred_llm = (np.asarray(p_llm) >= threshold).astype(int)
    tp = int(((pred_llm == 1) & (y_true == 1)).sum())
    fn = int(((pred_llm == 0) & (y_true == 1)).sum())
    fp = int(((pred_llm == 1) & (y_true == 0)).sum())
    tn = int(((pred_llm == 0) & (y_true == 0)).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    lr_plus = float("inf") if fpr == 0 and sens > 0 else (sens / fpr if fpr else 0.0)
    return lr_plus, sens, 1.0 - fpr, dict(tp=tp, fn=fn, fp=fp, tn=tn)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = _softmax(np.asarray(logits))
    preds = probs.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    lrp, sens, spec, _ = _lr_plus(labels, probs[:, 1], SUSPICIOUS_THRESHOLD)
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "lr_plus_suspicious": lrp,
        "sens_suspicious": sens,
        "spec_suspicious": spec,
    }


## 6. Train (resumable)

`save_strategy="epoch"` writes a checkpoint to Drive after every epoch.
The cell auto-detects an existing checkpoint and resumes from it, so a Colab
disconnect costs at most one epoch of progress.


In [ ]:
# Cell 6 — Trainer setup + resumable training.
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

# Resume if a checkpoint already exists on Drive.
import glob
_existing = sorted(glob.glob(str(CKPT_DIR / "checkpoint-*")))
_resume = bool(_existing)
print("resuming from checkpoint" if _resume else "starting fresh training")
trainer.train(resume_from_checkpoint=_resume)


## 7. Held-out evaluation — accuracy + LR+

The decisive numbers. Target **accuracy >= 0.85**. LR+ at the SUSPICIOUS
threshold tells you how much a SUSPICIOUS call shifts the odds — this is the
figure to quote in the changelog / docs (mirrors how T6/T7 are reported).


In [ ]:
# Cell 7 — evaluate on the held-out TEST split and print the headline metrics.
pred = trainer.predict(ds_test)
probs = _softmax(np.asarray(pred.predictions))
preds = probs.argmax(axis=1)
acc = accuracy_score(ds_test["label"], preds)
lrp, sens, spec, cm = _lr_plus(ds_test["label"], probs[:, 1], SUSPICIOUS_THRESHOLD)

print("=" * 56)
print(f"TEST accuracy (0.5 boundary) : {acc:.4f}   target >= 0.85 "
      f"-> {'PASS' if acc >= 0.85 else 'BELOW TARGET'}")
print(f"SUSPICIOUS threshold p(LLM)  : {SUSPICIOUS_THRESHOLD}")
print(f"  sensitivity                : {sens:.4f}")
print(f"  specificity                : {spec:.4f}")
print(f"  LR+                        : {lrp}")
print(f"  confusion @threshold       : {cm}")
print("=" * 56)


## 8. Export the model to Drive

Writes a standard `save_pretrained()` bundle (config + weights + tokenizer)
**and** a `t9_meta.json` recording the threshold and test metrics, so the T9
detector can load the operating point instead of hard-coding it.


In [ ]:
# Cell 8 — export model + tokenizer + metadata to Drive.
import json

trainer.save_model(str(EXPORT_DIR))      # config.json + model.safetensors
tokenizer.save_pretrained(str(EXPORT_DIR))

meta = {
    "base_model": BASE_MODEL,
    "max_len": MAX_LEN,
    "suspicious_threshold": SUSPICIOUS_THRESHOLD,
    "id2label": id2label,
    "test_accuracy": float(acc),
    "lr_plus_suspicious": (None if lrp == float("inf") else float(lrp)),
    "lr_plus_is_inf": lrp == float("inf"),
    "sensitivity_suspicious": float(sens),
    "specificity_suspicious": float(spec),
    "seed": SEED,
    "dataset": "Hello-SimpleAI/HC3 (all)",
}
(EXPORT_DIR / "t9_meta.json").write_text(json.dumps(meta, indent=2))
print("exported to:", EXPORT_DIR)
print("contents:", sorted(p.name for p in EXPORT_DIR.iterdir()))


## 9. Download & integrate into PaperGuard

### 9a. Get the model onto your machine

The bundle lives in your Drive at `MyDrive/paperguard_t9/t9_model/`. Either:

- **Drive sync / web**: download the whole `t9_model/` folder, or
- **zip it from Colab** (run the cell below) and download the single archive.

Then place the *contents* of `t9_model/` here:

```
~/.paperguard/models/t9/
|- config.json
|- model.safetensors
|- tokenizer_config.json
|- tokenizer.json (or vocab.txt)
|- special_tokens_map.json
|- t9_meta.json
```

On Windows that path is `C:\Users\<you>\.paperguard\models\t9\`.


In [ ]:
# Cell 9 (optional) — zip the export for a one-click download.
import shutil
from google.colab import files

archive = shutil.make_archive("/content/paperguard_t9_model", "zip", str(EXPORT_DIR))
print("archive:", archive)
files.download(archive)   # browser download prompt


### 9b. Add the T9 detector to PaperGuard

Once the model sits in `~/.paperguard/models/t9/`, create
`src/paperguard/detectors/t9_distilbert.py` with the template below, register
it like the other detectors, add tests, and ship as **3.0.0** (detector count
-> 41). T9 is **opt-in** (env var `PAPERGUARD_BERT_CHECK=1`) because it needs the
downloaded model and the `torch`/`transformers` extra — exactly the T7/T8
gating pattern.

```python
# src/paperguard/detectors/t9_distilbert.py
#
# T9 — DistilBERT learned LLM-text classifier (opt-in, needs local model).
#
# A *learned* complement to the statistical T7 (perplexity) and T8 (DetectGPT)
# detectors. Loads a DistilBERT fine-tuned on HC3 from
# ``~/.paperguard/models/t9/`` and reports the probability that a passage reads
# as ChatGPT-style text. Opt-in via ``PAPERGUARD_BERT_CHECK=1`` because it needs
# the downloaded weights and the torch/transformers extra.
#
# Iron rule: no verdict language; >= 3 innocent explanations on every Finding.
# The model emits a probability, never a confirmation.
from __future__ import annotations

import json
import logging
import os
import re
from pathlib import Path
from typing import Any, ClassVar

from paperguard.core.base_detector import BaseDetector
from paperguard.core.types import Finding, Severity

logger = logging.getLogger(__name__)

MODEL_DIR = Path.home() / ".paperguard" / "models" / "t9"
MIN_WORDS = 150
_DEFAULT_SUSPICIOUS = 0.90   # overridden by t9_meta.json if present
_CONCERN = 0.70              # NOTE < 0.70 <= CONCERN < SUSPICIOUS


def _opt_in_enabled() -> bool:
    return os.environ.get("PAPERGUARD_BERT_CHECK", "").lower() in {"1", "true", "yes"}


def _segment_text(text: str, max_chars: int = 2000) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    segs, buf = [], ""
    for s in sentences:
        if not s:
            continue
        if len(buf) + len(s) + 1 > max_chars and buf:
            segs.append(buf.strip()); buf = s
        else:
            buf = (buf + " " + s) if buf else s
    if buf:
        segs.append(buf.strip())
    return segs


class T9DistilBertDetector(BaseDetector):
    id: ClassVar[str] = "T9"
    name: ClassVar[str] = "DistilBERT LLM-text classifier"
    description: ClassVar[str] = (
        "Learned classifier (DistilBERT fine-tuned on HC3) estimating the "
        "probability that a passage reads as ChatGPT-style text."
    )
    academic_basis: ClassVar[str] = (
        "Guo et al. (2023) 'How Close is ChatGPT to Human Experts? "
        "Comparison Corpus and Detection' (HC3 dataset)."
    )
    data_requirements: ClassVar[list[str]] = ["manuscript_text"]
    # Shares the LLM-text assumption cluster with T6/T7/T8 — NOT independent
    # evidence; the combiner must not double-count them.
    assumption_cluster: ClassVar[str] = "llm_text_signature"

    _model: ClassVar[Any] = None
    _tokenizer: ClassVar[Any] = None
    _threshold: ClassVar[float] = _DEFAULT_SUSPICIOUS

    @classmethod
    def _load(cls) -> bool:
        if cls._model is not None:
            return True
        if not MODEL_DIR.exists():
            return False
        try:
            import torch  # noqa: F401
            from transformers import (
                AutoModelForSequenceClassification,
                AutoTokenizer,
            )
            cls._tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
            cls._model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
            cls._model.eval()
            meta_path = MODEL_DIR / "t9_meta.json"
            if meta_path.exists():
                meta = json.loads(meta_path.read_text())
                cls._threshold = float(meta.get("suspicious_threshold", _DEFAULT_SUSPICIOUS))
            return True
        except Exception as e:  # missing extra, corrupt weights, etc.
            logger.warning("T9 model load failed: %s", e)
            return False

    def check_applicability(self, data: Any) -> tuple[bool, str]:
        if not _opt_in_enabled():
            return False, "T9 is opt-in (set PAPERGUARD_BERT_CHECK=1)"
        if not isinstance(data, str):
            return False, "Expected text string"
        if len(re.findall(r"\b[a-zA-Z]+\b", data)) < MIN_WORDS:
            return False, "Text too short"
        if not self._load():
            return False, f"T9 model not found at {MODEL_DIR}"
        return True, ""

    def _detect(self, data: str, seed: int) -> list[Finding]:
        import torch

        segments = _segment_text(data)
        if not segments:
            return []
        probs: list[float] = []
        with torch.no_grad():
            for seg in segments:
                enc = self._tokenizer(
                    seg, truncation=True, max_length=512, return_tensors="pt"
                )
                logits = self._model(**enc).logits
                p_llm = torch.softmax(logits, dim=1)[0, 1].item()
                probs.append(p_llm)

        # Aggregate: the strongest segment drives the tier, but we report the
        # mean too so the reader sees how localized the signal is.
        p_max = max(probs)
        p_mean = sum(probs) / len(probs)

        if p_max >= self._threshold:
            sev = Severity.SUSPICIOUS
        elif p_max >= _CONCERN:
            sev = Severity.CONCERN
        elif p_max >= 0.50:
            sev = Severity.NOTE
        else:
            return []   # reads human-like; emit nothing

        return [
            Finding(
                detector_id=self.id,
                detector_name=self.name,
                severity=sev,
                summary=(
                    f"A passage scores p(LLM-style)={p_max:.2f} on the learned "
                    f"classifier"
                ),
                detail=(
                    "A DistilBERT model fine-tuned on the HC3 human-vs-ChatGPT "
                    f"corpus assigns a high LLM-style probability to "
                    f"{sum(p >= _CONCERN for p in probs)} of {len(segments)} "
                    "text segment(s). This is a stylistic similarity signal, "
                    "not a determination of authorship."
                ),
                evidence={
                    "p_llm_max": round(p_max, 4),
                    "p_llm_mean": round(p_mean, 4),
                    "n_segments": len(segments),
                    "threshold": self._threshold,
                },
                innocent_explanations=[
                    "The author is a non-native English writer whose phrasing "
                    "overlaps with the model's training distribution.",
                    "The passage was legitimately polished with an LLM writing "
                    "assistant, which journals increasingly permit when disclosed.",
                    "Formulaic sections (background, standard methods) read "
                    "'LLM-like' simply because the genre is highly conventional.",
                    "HC3 is ChatGPT-2023 era; domain or model drift can inflate "
                    "the score on perfectly human modern text.",
                ],
                academic_reference=self.academic_basis,
                applicability_notes=(
                    "Learned signal trained on HC3 (ChatGPT, 2023). Treat as a "
                    "screening prior, corroborate with T6/T7/T8 before acting."
                ),
            )
        ]
```

### 9c. Register, test, ship

1. Register in the detector registry next to T6/T7/T8 (same place they are
   wired up — grep for `t8_detectgpt` to find it).
2. Add `tests/test_t9_distilbert.py`: assert it skips cleanly when the model is
   absent and when `PAPERGUARD_BERT_CHECK` is unset (no GPU/model needed in CI).
3. Add the `torch`/`transformers` extra to `pyproject.toml` under an optional
   `[bert]` group so the core install stays light.
4. Update CHANGELOG, README detector roster (40 -> 41), and the HANDOFF docs.
5. Run the 3-gate (pytest -m "not network", ruff, mypy) and ship **3.0.0**.
